# Re-ID Evaluation — OSNet on Market-1501 and MSMT17

This notebook reproduces the standard re-ID benchmark numbers for the OSNet x1.0 model
pretrained on MSMT17. It runs in two stages:

1. **Market-1501** (~1.3 GB, ~10 min on T4) — quick sanity check. Expected: ~94.8 Rank-1 / 84.9 mAP.
2. **MSMT17** (~4 GB, ~40 min on T4) — the integrity gate. Expected: ~78.7 Rank-1 / 52.9 mAP.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.

---

## 1. Install

In [ ]:
# Install the trackers library from the feature branch + reid optional deps.
# The [reid] extra pulls in torch, torchvision, timm, and huggingface-hub.
!pip install -q --upgrade pip
!pip install -q 'git+https://github.com/roboflow/trackers.git@feat/reid-phase1#egg=trackers[reid]'

In [ ]:
import warnings
import numpy as np

# Confirm GPU is available.
import torch
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 2. Load the model

Downloads the OSNet x1.0 checkpoint pretrained on MSMT17 from Hugging Face (~6 MB).  
A domain warning is emitted — this is expected and intentional.

In [ ]:
from trackers.core.reid import ReIDModel, ReidEvaluator

model = ReIDModel.from_pretrained()   # downloads OSNet MSMT17 weights from HF
evaluator = ReidEvaluator(model, batch_size=256)  # T4 can handle 256 comfortably
print("Model loaded.")

---
## 3. Market-1501 — quick sanity check

**Expected numbers (OSNet x1.0 from the paper):** Rank-1 ≈ 94.8 %  |  mAP ≈ 84.9 %

Market-1501 is ~1.3 GB. We download it via gdown (official Google Drive mirror).
If gdown fails, see the alternative download cell below.

In [ ]:
!pip install -q gdown
import gdown, zipfile, os

MARKET_ZIP = "/content/Market-1501.zip"
MARKET_DIR = "/content/Market-1501-v15.09.15"

if not os.path.exists(MARKET_DIR):
    # Google Drive file ID for Market-1501-v15.09.15.zip
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=MARKET_ZIP, quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded.")

In [ ]:
# Alternative download if gdown fails (paste the unzip path that matches your mirror):
# !wget -q -O /content/Market-1501.zip "<YOUR_MIRROR_URL>"
# !unzip -q /content/Market-1501.zip -d /content

In [ ]:
from trackers.core.reid import load_market1501

query_m, gallery_m = load_market1501(MARKET_DIR)
print(f"Market-1501 — query: {len(query_m):,}  |  gallery: {len(gallery_m):,}")

In [ ]:
result_market = evaluator.evaluate(query_m, gallery_m)

print("\nMarket-1501 results:")
print(f"  mAP    : {result_market.metrics.map:.1f}%   (paper: ~84.9%)")
print(f"  Rank-1 : {result_market.metrics.rank1:.1f}%   (paper: ~94.8%)")
print(f"  Rank-5 : {result_market.metrics.rank5:.1f}%")
print(f"  Rank-10: {result_market.metrics.rank10:.1f}%")
print(f"  mINP   : {result_market.metrics.minp:.1f}%")

---
## 4. MSMT17 — full integrity gate

**Expected numbers (OSNet x1.0 from the paper):** Rank-1 ≈ 78.7 %  |  mAP ≈ 52.9 %

MSMT17 requires accepting the original dataset license:
http://www.pkuvmc.com/publications/msmt17.html

### Option A — download directly in Colab

If you have access to a hosted download link (e.g. from an academic institution),
paste it below.

In [ ]:
# Option A: direct URL (replace with your licensed download link)
# MSMT17_URL = "https://your-institution-mirror/MSMT17_V1.zip"
# !wget -q -O /content/MSMT17_V1.zip "$MSMT17_URL"
# !unzip -q /content/MSMT17_V1.zip -d /content

print("Skip this cell if using Option B (Google Drive mount).")

### Option B — mount from your Google Drive

1. Upload `MSMT17_V1.zip` (or the extracted folder) to your Google Drive.
2. Run the cell below to mount and set the path.

In [ ]:
# Option B: Google Drive
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile

# Adjust this path to wherever you placed the file in your Drive:
MSMT17_DRIVE_PATH = "/content/drive/MyDrive/datasets/MSMT17_V1.zip"
MSMT17_DIR = "/content/MSMT17_V1"

if not os.path.exists(MSMT17_DIR):
    print("Extracting MSMT17 (~4 GB, may take a few minutes)…")
    with zipfile.ZipFile(MSMT17_DRIVE_PATH, "r") as zf:
        zf.extractall("/content")
    print("Done.")
else:
    print("Already extracted.")

In [ ]:
# If your directory is already extracted and not a zip (Option A direct download),
# set MSMT17_DIR to that path here:
# MSMT17_DIR = "/content/MSMT17_V1"

from trackers.core.reid import load_msmt17

query_ms, gallery_ms = load_msmt17(MSMT17_DIR)
print(f"MSMT17 — query: {len(query_ms):,}  |  gallery: {len(gallery_ms):,}")

In [ ]:
result_msmt17 = evaluator.evaluate(query_ms, gallery_ms)

print("\nMSMT17 results:")
print(f"  mAP    : {result_msmt17.metrics.map:.1f}%   (paper: ~52.9%)")
print(f"  Rank-1 : {result_msmt17.metrics.rank1:.1f}%   (paper: ~78.7%)")
print(f"  Rank-5 : {result_msmt17.metrics.rank5:.1f}%")
print(f"  Rank-10: {result_msmt17.metrics.rank10:.1f}%")
print(f"  mINP   : {result_msmt17.metrics.minp:.1f}%")

---
## 5. Summary table

In [ ]:
print(f"{'Dataset':<14} {'mAP':>8} {'Rank-1':>8} {'Rank-5':>8} {'Rank-10':>9} {'mINP':>8}")
print("-" * 55)

def _row(name, m):
    return f"{name:<14} {m.map:>7.1f}% {m.rank1:>7.1f}% {m.rank5:>7.1f}% {m.rank10:>8.1f}% {m.minp:>7.1f}%"

print(_row("Market-1501", result_market.metrics))
print(_row("MSMT17",      result_msmt17.metrics))
print("-" * 55)
print("Paper targets: Market-1501 R1≈94.8 mAP≈84.9  |  MSMT17 R1≈78.7 mAP≈52.9")